In [5]:
import pandas as pd
import numpy as np

# Datensatz laden (nur train für schnellen Überblick)
train_df = pd.read_csv("datasets/CICIOT23/train/train.csv")
val_df   = pd.read_csv("datasets/CICIOT23/validation/validation.csv")
test_df  = pd.read_csv("datasets/CICIOT23/test/test.csv")

print("Shape Train:", train_df.shape)
print("Shape Val:  ", val_df.shape)
print("Shape Test: ", test_df.shape)

# Alle Spaltennamen anzeigen
print("\nSpalten:\n", train_df.columns.tolist())

# Erste Zeilen
train_df.head()

Shape Train: (5491971, 47)
Shape Val:   (1176851, 47)
Shape Test:  (1176851, 47)

Spalten:
 ['flow_duration', 'Header_Length', 'Protocol Type', 'Duration', 'Rate', 'Srate', 'Drate', 'fin_flag_number', 'syn_flag_number', 'rst_flag_number', 'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number', 'ack_count', 'syn_count', 'fin_count', 'urg_count', 'rst_count', 'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP', 'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number', 'Magnitue', 'Radius', 'Covariance', 'Variance', 'Weight', 'label']


,flow_duration,Header_Length,Protocol Type,Duration,Rate,Srate,Drate,fin_flag_number,syn_flag_number,rst_flag_number,...,Std,Tot size,IAT,Number,Magnitue,Radius,Covariance,Variance,Weight,label
0,0.000000,757.00,6.00,64.00,23.671858,23.671858,0.0,0.0,0.0,0.0,...,538.470740,944.00,8.334058e+07,9.5,41.845546,761.456760,305219.322301,0.95,141.55,DDoS-ACK_Fragmentation
1,0.000000,54.00,6.00,64.00,2.393046,2.393046,0.0,0.0,1.0,0.0,...,0.000000,54.00,8.309327e+07,9.5,10.392305,0.000000,0.000000,0.00,141.55,DDoS-SYN_Flood
2,0.033982,56.78,6.11,64.64,1.192715,1.192715,0.0,0.0,0.0,0.0,...,1.727526,54.29,8.333086e+07,9.5,10.462813,2.445286,16.853118,0.19,141.55,DDoS-PSHACK_Flood
3,0.000000,0.00,47.00,64.00,9.841972,9.841972,0.0,0.0,0.0,0.0,...,0.000000,592.00,8.370278e+07,9.5,34.409301,0.000000,0.000000,0.00,141.55,Mirai-greeth_flood
4,3.944828,108.00,6.00,64.00,0.506993,0.506993,0.0,0.0,1.0,0.0,...,0.000000,54.00,8.297270e+07,9.5,10.392305,0.000000,0.000000,0.00,141.55,DoS-SYN_Flood


In [10]:
for col in ['flow_duration', 'Duration', 'IAT']:
    if col in train_df.columns:
        print(f"\n{'='*40}")
        print(f"Column: {col}")
        
        # cumulative sum over entire column
        cumsum = train_df[col].cumsum()
        
        # check if column is monotonically increasing
        is_sorted = train_df[col].is_monotonic_increasing
        print(f"Monotonically increasing: {is_sorted}")
        
        # first and last values
        print(f"\nFirst 10 values: {train_df[col].head(10).tolist()}")
        print(f"Last  10 values: {train_df[col].tail(10).tolist()}")
        
        # step sizes between consecutive rows
        diffs = train_df[col].diff().dropna()
        print(f"\nStep sizes (diff):")
        print(f"  Min:              {diffs.min():.4f}")
        print(f"  Max:              {diffs.max():.4f}")
        print(f"  Median:           {diffs.median():.4f}")
        print(f"  Share of negative steps (backwards jumps): {(diffs < 0).mean():.2%}")
        
        # calculate indices separately to avoid f-string syntax issues
        idx_10 = int(len(cumsum) * 0.1)
        idx_50 = int(len(cumsum) * 0.5)
        
        print(f"\nCumulative progression:")
        print(f"  After  10% of data: {cumsum.iloc[idx_10]:.2f}")
        print(f"  After  50% of data: {cumsum.iloc[idx_50]:.2f}")
        print(f"  After 100% of data: {cumsum.iloc[-1]:.2f}")


Column: flow_duration
Monotonically increasing: False

First 10 values: [0.0, 0.0, 0.0339823198318481, 0.0, 3.944827942848205, 0.0, 0.0, 0.0, 0.0, 0.0]
Last  10 values: [0.8047924900054931, 0.0, 0.0, 0.1054975128173828, 0.324876012802124, 0.0, 0.0, 0.0, 0.0, 0.0]

Step sizes (diff):
  Min:              -99435.6792
  Max:              99435.7618
  Median:           0.0000
  Share of negative steps (backwards jumps): 35.30%

Cumulative progression:
  After  10% of data: 2975832.30
  After  50% of data: 15304181.98
  After 100% of data: 31069972.81

Column: Duration
Monotonically increasing: False

First 10 values: [64.0, 64.0, 64.64, 64.0, 64.0, 64.0, 64.0, 64.0, 64.0, 64.0]
Last  10 values: [64.0, 64.0, 64.0, 64.0, 64.0, 64.0, 64.0, 64.0, 64.59, 64.0]

Step sizes (diff):
  Min:              -224.2000
  Max:              201.7000
  Median:           0.0000
  Share of negative steps (backwards jumps): 22.82%

Cumulative progression:
  After  10% of data: 36425602.86
  After  50% of data:

In [11]:
# Schwierig hier weiterzu arbeiten da wir keine ordentliche Timewindows gestalten können